# News NLP pipeline — small runnable demo

This notebook follows a handful of synthetic headlines through the same stages used by the prototype. There is no live news retrieval, and the sample does not state real events about real companies.

![News NLP pipeline](../docs/news_nlp/pipeline_overview.svg)

By default, tiny deterministic test doubles exercise the plumbing without downloading model weights. Set `RUN_REAL_INFERENCE = True` to use FinBERT-ESG, the MiniLM financial fallback and our fine-tuned direction checkpoint.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise FileNotFoundError("Start Jupyter in the repository root or notebooks/ folder.")

sys.path.insert(0, str(REPO_ROOT))
from src.news_nlp import (
    NewsPipeline, aggregate_event_features, build_default_pipeline,
    compute_tone_momentum,
)

## 1. Load the demonstration headlines

These ten rows were written for this repository. Two rows deliberately repeat one headline so we can see that duplicate coverage does not create two direction votes.

In [ ]:
AS_OF_UTC = "2026-09-13T00:00:00Z"
RUN_REAL_INFERENCE = False
DIRECTION_MODEL_PATH_OR_ID = os.environ.get("DIRECTION_MODEL_PATH_OR_ID", "")

news = pd.read_csv(REPO_ROOT / "data" / "news_demo_headlines.csv")
assert news["is_synthetic"].eq(1).all()
assert news[["company_id", "company_name", "headline", "published_at_utc"]].notna().all().all()

display(news[["company_name", "headline", "published_at_utc"]])

## 2. Choose real models or a plumbing check

The test doubles below are deliberately simple keyword rules. Their outputs are **not model results and not an accuracy claim**; they only let reviewers run every pipeline step offline. The real path uses the three model layers.

In [ ]:
class DemoESGClassifier:
    # Deterministic, plumbing-only replacement for FinBERT-ESG.
    def predict_proba(self, texts):
        answers = []
        for text in texts:
            text = text.lower()
            if any(word in text for word in ("solar", "emissions")):
                top = "environmental"
            elif any(word in text for word in ("safety", "parental leave")):
                top = "social"
            elif any(word in text for word in ("directors", "disclosure", "meeting")):
                top = "governance"
            else:
                top = "none"
            result = {label: 0.025 for label in ("environmental", "social", "governance", "none")}
            result[top] = 0.925
            answers.append(result)
        return answers


class DemoFinancialFallback:
    # Plumbing-only replacement for the MiniLM fallback.
    def score(self, texts):
        terms = ("profit", "loan-loss", "defaults")
        return [0.85 if any(term in text.lower() for term in terms) else None for text in texts]


class DemoDirectionClassifier:
    # Plumbing-only replacement for the fine-tuned direction model.
    def predict_proba(self, texts):
        answers = []
        for text in texts:
            text = text.lower()
            negative = ("increase in", "raises", "recalls", "inquiry")
            positive = ("begins operating", "expands", "appoints")
            top = "negative" if any(word in text for word in negative) else (
                "positive" if any(word in text for word in positive) else "neutral"
            )
            result = {label: 0.10 for label in ("negative", "neutral", "positive")}
            result[top] = 0.80
            answers.append(result)
        return answers

In [ ]:
if RUN_REAL_INFERENCE:
    if not DIRECTION_MODEL_PATH_OR_ID:
        raise ValueError("Set DIRECTION_MODEL_PATH_OR_ID to the selected checkpoint.")
    pipeline = build_default_pipeline(DIRECTION_MODEL_PATH_OR_ID, device="auto")
    run_label = "real model inference"
else:
    pipeline = NewsPipeline(
        esg_classifier=DemoESGClassifier(),
        financial_fallback=DemoFinancialFallback(),
        direction_classifier=DemoDirectionClassifier(),
    )
    run_label = "offline plumbing check — test doubles, not model predictions"

print(run_label)

## 3. Classify and inspect the trace

Company names are masked before topic classification. FinBERT-ESG chooses Environmental, Social, Governance or None. Only None can enter the conservative financial fallback. The assigned pillar then becomes part of the direction model's input.

In [ ]:
scored = pipeline.classify(news, deduplicate=True)

trace_columns = [
    "company_name", "headline", "article_count", "esg_top_label",
    "pillar_label", "pillar_method", "financial_fallback_score",
    "direction_label", "p_negative", "p_neutral", "p_positive",
    "signed_tone", "inference_status",
]
display(scored[trace_columns].round(3))

assert len(scored) == len(news) - 1  # the exact duplicate became one event
assert scored["signed_tone"].dropna().between(-1, 1).all()
fallback_rows = scored["pillar_method"].eq("minilm_financial_fallback")
assert fallback_rows.any()
assert scored.loc[fallback_rows, "esg_top_label"].eq("none").all()

## 4. Build company features

For each company, pillar and window, the feature builder counts deduplicated events and computes weighted tone. Direction is

$$s_i=P_i(\text{positive})-P_i(\text{negative}),$$

and recent evidence receives more weight. Thin evidence is shrunk toward zero, while raw tone remains missing when there is no evidence.

In [ ]:
features = aggregate_event_features(
    scored,
    company_ids=news["company_id"].unique(),
    as_of_utc=AS_OF_UTC,
    windows=(30, 90, 180),
)
momentum = compute_tone_momentum(features)

summary_90d = features.loc[
    features["window_days"].eq(90) & features["event_count"].gt(0),
    ["company_id", "pillar", "event_count", "article_count", "effective_weight",
     "tone_raw", "tone_shrunk_to_zero"],
].sort_values(["company_id", "pillar"])

display(summary_90d.round(3))
display(momentum.loc[momentum["baseline_window_has_evidence"].eq(1)].round(3))
assert len(features) == news["company_id"].nunique() * 4 * 3

## Reading the output

- `pillar_method` shows whether FinBERT-ESG or the financial fallback assigned the topic.
- `signed_tone` is continuous: negative values point toward adverse news and positive values toward favorable news.
- Counts describe the collected sample; they are not company ESG performance.
- This notebook proves the inference and aggregation path. The full prototype still needs dated retrieval, relevance review and coverage diagnostics before its features are merged with structured ESG data.